# Hardware-ground-truth capture protocol

This notebook is a field worksheet for validating synchronization against a physical event. A timestamp report measures internal consistency; an LED flash plus an audio beep provides independent evidence of when the event happened. Run the sequence at the start and end of every capture session.

**Safety and rights:** use a visible, non-hazardous LED; record only people and spaces for which you have permission; keep raw media and consent records with the run. Do not publish identifiable video or audio as part of a fixture.

## 1. Equipment and event

Use one bright LED visible to every camera and one short electronic beep audible to every microphone. Drive both from the same event controller when possible. Place the LED near the task volume, keep every lens unobstructed, and avoid automatic exposure changes during the event.

Capture 3–5 pulses, separated by several seconds. Include one pulse near the beginning and one near the end of the recording. Also record the device serial numbers, firmware, frame/audio rates, exposure settings, trigger configuration, and every clock domain.

## 2. Compare at least three clock/capture setups

| Setup | Capture path | What it tests | Baseline |
|---|---|---|---|
| A — hardware synchronized | shared trigger, genlock, or common clock; LED/beep controller timestamped too | sensor and recorder behavior when clocks share hardware | hardware-synchronized |
| B — software-only, shared host | independent sensors timestamped by one host clock on arrival | callback, transport, and host scheduling offset/jitter | software-only |
| C — software-only, independent clocks | device clocks mapped after capture; no shared trigger | clock offset, drift, reconnect/reset behavior | software-only |

Keep the physical event and scene fixed across setups. If possible, repeat each setup twice: the start/end difference is the drift estimate. A hardware trigger is a capture aid, not by itself an acceptance result.

## 3. Capture checklist

- [ ] Save original camera and microphone media without transcoding.
- [ ] Save device timestamps, host receive timestamps, clock-domain names, and reset/epoch counters.
- [ ] Log the setup (A, B, or C), trigger wiring, rates, exposure, audio gain, and controller ID.
- [ ] Make 3–5 LED+beep pulses at the start and repeat them at the end.
- [ ] Record any dropped frames, audio clipping, occlusion, rolling-shutter concerns, or reconnects.
- [ ] Hash raw files and retain a manifest; publish only a redacted or synthetic derivative unless redistribution rights are explicit.

## 4. Analysis and acceptance

For each camera–microphone pair, estimate the LED onset in video and beep onset in audio for every pulse. Store `offset_ns`, `drift_ppb`, `anchor_time_ns`, variance, detector settings, and rejected events. Compare those estimates with the device-timestamp mapping and with `embodied-sync`'s report.

An acceptable result is defined before analysis: every required stream has a missing rate below its threshold, absolute skew below its threshold, and start/end physical estimates that agree within the stated uncertainty. An unacceptable result includes a timestamp report that looks clean while the LED/beep estimate disagrees, a clock reset, or unexplained setup disagreement. Do not average away disagreement.

Interpretation: agreement between physical methods and device timestamps supports a bounded assurance claim. Agreement between physical methods but disagreement with device timestamps indicates a constant pipeline latency. A start/end change indicates drift; fit offset plus drift rather than applying one constant correction.

In [ ]:
# Fill these in before collecting data. Values are examples in milliseconds.
threshold_ms = {"camera": 20.0, "microphone": 10.0}
physical_start_ms = {"camera-microphone": 3.2}
physical_end_ms = {"camera-microphone": 4.1}

for pair, start in physical_start_ms.items():
    end = physical_end_ms[pair]
    print(f"{pair}: start={start:.2f} ms, end={end:.2f} ms, drift={end-start:+.2f} ms")
print("Record the detector uncertainty and per-stream report values before accepting the run.")

## 5. Record the result

Use [`docs/acceptance_report_template.md`](../../docs/acceptance_report_template.md) for the signed result. Include the generated HTML and JSON reports, this worksheet's setup table, raw-event IDs, start/end estimates, uncertainty, and a short explanation for every rejected or borderline event.

Remember the distinction: hardware synchronization makes simultaneous capture more likely; synchronization assurance demonstrates what the recorded streams and independent physical checks support. Offline alignment can use future samples, while a causal online policy may use only samples already received.